In [1]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
import torch

processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

C:\Users\bmaks\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 473/473 [00:00<00:00, 34105.31it/s]


In [6]:
image = Image.open("house_images\\all_images\\all_images\\008_88b44079.jpg").convert("RGB")

inputs = processor(image, return_tensors="pt")
out = model.generate(**inputs)
description = processor.decode(out[0], skip_special_tokens=True)

print(f"Описание: {description}")

Описание: a modern house with stairs leading up to the front door


In [9]:
import os
import pandas as pd

IMAGE_DIR = "house_images/all_images/all_images"
LABELS_CSV = "house_images/labels.csv"
OUTPUT_CSV = "architectural_descriptions.csv"

labels_df = pd.read_csv(LABELS_CSV)
labels_map = dict(zip(labels_df['file_label'], labels_df['house_type']))

data = []
image_files = [f for f in os.listdir(IMAGE_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

for i, filename in enumerate(image_files):
    house_type = labels_map.get(filename, "unknown")
    img_path = os.path.join(IMAGE_DIR, filename)
    image = Image.open(img_path).convert("RGB")
    
    inputs = processor(image, return_tensors="pt")
    out = model.generate(**inputs)
    description = processor.decode(out[0], skip_special_tokens=True)

    data.append({
        "filename": filename,
        "text": description,
        "label": house_type
    })
    if (i + 1) % 10 == 0:
        print(f"Done: {i+1}/{len(image_files)}")

result_df = pd.DataFrame(data)
result_df.to_csv(OUTPUT_CSV, index=False)
print(result_df.head())

Done: 10/867
Done: 20/867
Done: 30/867
Done: 40/867
Done: 50/867
Done: 60/867
Done: 70/867
Done: 80/867
Done: 90/867
Done: 100/867
Done: 110/867
Done: 120/867
Done: 130/867
Done: 140/867
Done: 150/867
Done: 160/867
Done: 170/867
Done: 180/867
Done: 190/867
Done: 200/867
Done: 210/867
Done: 220/867
Done: 230/867
Done: 240/867
Done: 250/867
Done: 260/867
Done: 270/867
Done: 280/867
Done: 290/867
Done: 300/867
Done: 310/867
Done: 320/867
Done: 330/867
Done: 340/867
Done: 350/867
Done: 360/867
Done: 370/867
Done: 380/867
Done: 390/867
Done: 400/867
Done: 410/867
Done: 420/867
Done: 430/867
Done: 440/867
Done: 450/867
Done: 460/867
Done: 470/867
Done: 480/867
Done: 490/867
Done: 500/867
Done: 510/867
Done: 520/867
Done: 530/867
Done: 540/867
Done: 550/867
Done: 560/867
Done: 570/867
Done: 580/867
Done: 590/867
Done: 600/867
Done: 610/867
Done: 620/867
Done: 630/867
Done: 640/867
Done: 650/867
Done: 660/867
Done: 670/867
Done: 680/867
Done: 690/867
Done: 700/867
Done: 710/867
Done: 720/867
D

In [34]:
import pandas as pd
from sklearn.model_selection import train_test_split
from collections import Counter

df = pd.read_csv("architectural_descriptions.csv")
print(f"Исходный объем: {len(df)} записей")

initial_count = len(df)
df.drop_duplicates(subset=['text'], inplace=True)
exact_dups_removed = initial_count - len(df)
print(f"Удалено дубликатов: {exact_dups_removed}")

print(f"Итоговый объем: {len(df)} записей")

print("\nРаспределение классов:")
class_counts = Counter(df['label'])
for cls, count in sorted(class_counts.items()):
    pct = count / len(df) * 100
    print(f"  {cls}: {count} ({pct:.1f}%)")

train_df, temp_df = train_test_split(
    df, test_size=0.95, random_state=42, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.8, random_state=42, stratify=temp_df['label']
)

print(f"Train: {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)")
print(f"Val:   {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)")
print(f"Test:  {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)")

train_df.to_csv("data/train.csv", index=False)
val_df.to_csv("data/val.csv", index=False)
test_df.to_csv("data/test.csv", index=False)

Исходный объем: 867 записей
Удалено дубликатов: 340
Итоговый объем: 527 записей

Распределение классов:
  farmhouse: 257 (48.8%)
  modern: 148 (28.1%)
  rustic: 122 (23.1%)
Train: 26 (4.9%)
Val:   100 (19.0%)
Test:  401 (76.1%)


In [38]:
import os, json, time, random
import pandas as pd
from llama_cpp import Llama
from sklearn.metrics import accuracy_score, f1_score

MODELS = {
    "Qwen2.5-1.5B": r"C:\local_models\qwen2.5-1.5b-instruct-q4_k_m.gguf",
    "SmolLM2-1.7B": r"C:\local_models\smollm2-1.7b-instruct-q4_k_m.gguf",
    "Llama-3.2-3B": r"C:\local_models\Llama-3.2-3B-Instruct-Q4_K_M.gguf",
}

TRAIN_CSV = "data/train.csv"
VAL_CSV = "data/test.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df = pd.read_csv(VAL_CSV)

unique_classes = sorted(train_df['label'].unique().tolist())
classes_str = ", ".join(unique_classes)

class_counts = train_df['label'].value_counts()

SYSTEM_PROMPT = f"""Ты — эксперт по архитектуре. Определи тип здания по описанию.
Ответь ТОЛЬКО в формате JSON: {{"type": "..."}}
Допустимые значения type: {classes_str}."""

def get_few_shot():   
    return train_df

def build_prompt(text, examples=None):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    if examples is not None and len(examples) > 0:
        iter_examples = examples.to_dict('records')
            
        for ex in iter_examples:
            msgs.append({"role": "user", "content": ex["text"]})
            msgs.append({
                "role": "assistant", 
                "content": json.dumps({"type": ex["label"]}, ensure_ascii=False)
            })
            
    msgs.append({"role": "user", "content": text})
    return msgs

def parse_json(raw):
    try:
        s, e = raw.find("{"), raw.rfind("}") + 1
        return json.loads(raw[s:e]) if s != -1 and e > 0 else None
    except: 
        return None

few_shot = get_few_shot()
results = []

for name, path in MODELS.items():
        
    print(f"Модель: {name}")
    llm = Llama(model_path=path, n_ctx=2048, n_threads=4, verbose=False)
    
    preds, lats, invalid = [], [], 0
    
    for _, row in val_df.iterrows():
        t0 = time.time()
        out = llm.create_chat_completion(
            messages=build_prompt(row["text"], few_shot), 
            temperature=0.0, max_tokens=64
        )
        lat = (time.time() - t0) * 1000
        lats.append(lat)
        
        parsed = parse_json(out["choices"][0]["message"]["content"].strip())
        if parsed and parsed.get("type") in unique_classes:
            preds.append(parsed["type"])
        else:
            preds.append(None); invalid += 1
            
    true = val_df["label"].tolist()
    valid_mask = [p is not None for p in preds]
    
    y_true_valid = [t for t, v in zip(true, valid_mask) if v]
    y_pred_valid = [p for p, v in zip(preds, valid_mask) if v]
    
    acc = accuracy_score(y_true_valid, y_pred_valid)
    macro_f1 = f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0)
    
    results.append({
        "Model": name, 
        "Accuracy": round(acc, 4), 
        "Macro-F1": round(macro_f1, 4),
        "Invalid%": round(invalid / len(val_df) * 100, 1),
        "AvgLatency_ms": round(sum(lats)/len(lats), 1),
        "P95Latency_ms": round(sorted(lats)[int(len(lats)*0.95)], 1)
    })
    del llm

df_res = pd.DataFrame(results).sort_values("Macro-F1", ascending=False)
print(df_res.to_string(index=False))
df_res.to_csv("experiment_results_local.csv", index=False)

Модель: Qwen2.5-1.5B
Модель: SmolLM2-1.7B
Модель: Llama-3.2-3B
       Model  Accuracy  Macro-F1  Invalid%  AvgLatency_ms  P95Latency_ms
Llama-3.2-3B    0.7825    0.7748       0.2          875.5         1019.3
Qwen2.5-1.5B    0.7406    0.7357       0.0          415.9          475.6
SmolLM2-1.7B    0.6842    0.6226       0.5          522.9          548.0


In [39]:
MODELS = {
    "Llama-3.2-3B": r"C:\local_models\Llama-3.2-3B-Instruct-Q4_K_M.gguf"
}

TEST_CSV = "data/test.csv"
test_df = pd.read_csv(TEST_CSV)

unique_classes = sorted(train_df['label'].unique().tolist())
classes_str = ", ".join(unique_classes)

class_counts = train_df['label'].value_counts()

SYSTEM_PROMPT = f"""Ты — эксперт по архитектуре. Определи тип здания по описанию.
Ответь ТОЛЬКО в формате JSON: {{"type": "..."}}
Допустимые значения type: {classes_str}."""

def get_few_shot():   
    return train_df

def build_prompt(text, examples=None):
    msgs = [{"role": "system", "content": SYSTEM_PROMPT}]
    
    if examples is not None and len(examples) > 0:
        iter_examples = examples.to_dict('records')
            
        for ex in iter_examples:
            msgs.append({"role": "user", "content": ex["text"]})
            msgs.append({
                "role": "assistant", 
                "content": json.dumps({"type": ex["label"]}, ensure_ascii=False)
            })
            
    msgs.append({"role": "user", "content": text})
    return msgs

def parse_json(raw):
    try:
        s, e = raw.find("{"), raw.rfind("}") + 1
        return json.loads(raw[s:e]) if s != -1 and e > 0 else None
    except: 
        return None

few_shot = get_few_shot()
results = []

for name, path in MODELS.items():
        
    print(f"Модель: {name}")
    llm = Llama(model_path=path, n_ctx=2048, n_threads=4, verbose=False)
    
    preds, lats, invalid = [], [], 0
    
    for _, row in test_df.iterrows():
        t0 = time.time()
        out = llm.create_chat_completion(
            messages=build_prompt(row["text"], few_shot), 
            temperature=0.0, max_tokens=64
        )
        lat = (time.time() - t0) * 1000
        lats.append(lat)
        
        parsed = parse_json(out["choices"][0]["message"]["content"].strip())
        if parsed and parsed.get("type") in unique_classes:
            preds.append(parsed["type"])
        else:
            preds.append(None); invalid += 1
            
    true = test_df["label"].tolist()
    valid_mask = [p is not None for p in preds]
    
    y_true_valid = [t for t, v in zip(true, valid_mask) if v]
    y_pred_valid = [p for p, v in zip(preds, valid_mask) if v]
    
    acc = accuracy_score(y_true_valid, y_pred_valid)
    macro_f1 = f1_score(y_true_valid, y_pred_valid, average="macro", zero_division=0)
    
    results.append({
        "Model": name, 
        "Accuracy": round(acc, 4), 
        "Macro-F1": round(macro_f1, 4),
        "Invalid%": round(invalid / len(test_df) * 100, 1),
        "AvgLatency_ms": round(sum(lats)/len(lats), 1),
        "P95Latency_ms": round(sorted(lats)[int(len(lats)*0.95)], 1)
    })
    del llm

df_res = pd.DataFrame(results).sort_values("Macro-F1", ascending=False)
print(df_res.to_string(index=False))
df_res.to_csv("experiment_results_local_test.csv", index=False)

Модель: Llama-3.2-3B
       Model  Accuracy  Macro-F1  Invalid%  AvgLatency_ms  P95Latency_ms
Llama-3.2-3B    0.7825    0.7748       0.2          889.6         1000.9
